# Fourier Locking in DRU-PQCs — paper experiments

Companion notebook to *Overcoming Fourier Locking in Quantum Data Re-uploading
Classifiers via Spectral Homotopy* (Topel, submitted to PRX Quantum). Contains
the experiments underlying the reported statistics.

License: TODO — set before public release (e.g. CC-BY-4.0 for content;
MIT/Apache-2.0 for the code files if separated).

## Experiments

1. **Diagnostic tracking** — 4-qubit DRU-PQC at $f=3.0$, N=50 seeds each in STD
   and ID conditions. Tracks the classical FDR, the parameter-space QFIM scalar,
   and the input-space QFI $F_x$ at milestone epochs. Correlations against the
   binary escape outcome underlie Fig. 1 and Sec. V.A. Writes
   `experiment_a_results.json`.

2. **Routing ablation** — STD vs. identity-initialized (ID) vs.
   coupled-warmup (COUP) entanglers. Fig. 2. Writes
   `experiment_b_results_clean.json`.

3. **Curriculum ablation** — four training curricula (STD / FWD / REV / RND),
   three 40-epoch phases each. Per-phase accuracy, parameter-space QFIM, and
   input-space $F_x$. Figs. 3 and 4. Writes
   `experiment_f_bulletproof_qfi.json`.

4. **Reproduction gate** — verifies that adding QFI logging did not perturb
   training (final accuracies match bit-for-bit) before merging QFI traces into
   `experiment_f_bulletproof.json`.

5. **Fisher exact tests** — significance tests on the curriculum contingency
   tables (Sec. V.B).

6. **FWD trajectory analysis** — per-phase accuracy trajectories for the
   successful FWD seeds and the Phase-1 fitters under REV.

## Reproducibility

4-qubit DRU-PQC, L=2 layers. Batch size 256, Adam at lr=0.01. 4000 samples
uniform on $[-\pi, \pi]^4$. Escape threshold 0.65 test accuracy. N=50 seeds.

## Fisher quantities

Three quantities are computed and stored under distinct keys — they measure
different things and are not interchangeable:

- `fisher_traj` — classical Fisher discriminant ratio of the PQC output
  features (between-class over within-class variance). Diagnoses readout
  collapse.
- `qfim_traj` / `phase_qfi` — parameter-space QFIM scalar
  $\frac{1}{N}\mathrm{Tr}(F)$, from the state-vector Jacobian (paper Eq. 2).
- `xqfi_traj` / `phase_xqfi` — input-space QFI $F_x$, the Fubini–Study
  susceptibility of the state to the encoded input.

Both quantum-geometric quantities are evaluated analytically on a fixed probe
batch (the first 8 validation inputs), consume no RNG, and do not perturb
training.

## Experiment 1 — Diagnostic tracking (Fig. 1, Sec. V.A)

Trains 50 STD-condition and 50 ID-condition seeds to convergence on the $f=3.0$
product-of-sines classifier. At the milestone epochs $[0, 10, 30, 60, 120]$,
records three quantities per seed:

1. `fisher_traj` — classical Fisher discriminant ratio of the PQC output
   features. **This is what Fig. 1 plots** ($r_{pb} = 0.912$).
2. `qfim_traj` — parameter-space QFIM scalar $\frac{1}{N}\mathrm{Tr}(F)$,
   from the state-vector Jacobian. Diagnostically null ($r_{pb} \approx 0.10$).
3. `xqfi_traj` — input-space QFI $F_x$, summed over the four input dimensions.
   Its trajectory (not endpoint) carries the migration signature.

The analysis block at the end prints point-biserial correlations of all three
quantities against the binary escape outcome and the Spearman correlation
between initial $\omega^2$ and initial $F_x$.

Output: `experiment_a_results.json`.

In [1]:
import pennylane as qml
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
from scipy import stats
import math
import json

# --- Hyperparameters ---
SAMPLES = 4000
FREQ = 3.0
EPOCHS = 120
BATCH_SIZE = 256
LR = 0.01
SEEDS = 50
TRACK_EPOCHS = [0, 10, 30, 60, 120] # Milestones for Fisher Score tracking

n_qubits = 4
n_layers = 2

# --- Model and helpers ---
def generate_data(samples, freq):
    X = (torch.rand(samples, 4) * 2 * math.pi) - math.pi
    y_raw = torch.sin(freq * X[:, 0]) * torch.sin(freq * X[:, 1]) * \
            torch.sin(freq * X[:, 2]) * torch.sin(freq * X[:, 3])
    y = (y_raw > 0).long()
    return X, y

def make_pqc():
    dev = qml.device("default.qubit", wires=n_qubits)
    @qml.qnode(dev, interface="torch")
    def qc(inputs, weights):
        for l in range(n_layers):
            for i in range(n_qubits):
                qml.RX(weights[l, i, 0] * inputs[:, i] + weights[l, i, 1], wires=i)
            qml.StronglyEntanglingLayers(weights[l:l+1, :, 2:5], wires=range(n_qubits))
        return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]
        
    pqc_layer = qml.qnn.TorchLayer(qc, {"weights": (n_layers, n_qubits, 5)})
    return nn.Sequential(pqc_layer, nn.Linear(n_qubits, 2))

def apply_routing_identity_init(model):
    """Initialize entangling angles near zero (approximate identity), leaving
    encoding weights untouched."""
    with torch.no_grad():
        for name, param in model.named_parameters():
            if "weights" in name:
                # Target only the StronglyEntanglingLayers angles (indices 2, 3, 4)
                torch.nn.init.normal_(param[:, :, 2:5], mean=0.0, std=0.01)

def compute_fisher_score(model, val_loader):
    """Classical Fisher discriminant ratio of the PQC output features."""
    model.eval()
    with torch.no_grad():
        pqc_out_class0, pqc_out_class1 = [], []
        for bx, by in val_loader:
            out = model[0](bx)
            pqc_out_class0.append(out[by == 0])
            pqc_out_class1.append(out[by == 1])
            
        out0 = torch.cat(pqc_out_class0)
        out1 = torch.cat(pqc_out_class1)
        
        mean_0, mean_1 = out0.mean(dim=0), out1.mean(dim=0)
        between = (mean_1 - mean_0) ** 2
        within = (out0.var(dim=0) + out1.var(dim=0)) / 2
        fisher = between / (within + 1e-8)
        
    return fisher.sum().item()

# --- Parameter-space QFIM: 1/N Tr(F), paper Eq. (2) ---
# Distinct from compute_fisher_score (classical FDR of the outputs).
# Computes F_ii = 4(<d_i psi|d_i psi> - |<psi|d_i psi>|^2) from the
# state-vector Jacobian. Deterministic (analytic backprop), consumes no RNG.

_qfim_dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(_qfim_dev, interface="torch", diff_method="backprop")
def _state_circuit(x, weights):
    for l in range(n_layers):
        for i in range(n_qubits):
            qml.RX(weights[l, i, 0] * x[i] + weights[l, i, 1], wires=i)
        qml.StronglyEntanglingLayers(weights[l:l+1, :, 2:5], wires=range(n_qubits))
    return qml.state()

def qfim_trace_scalar(model, X_probe, n_total_params=50):
    """Normalized QFIM trace (1/N Tr(F)) averaged over a fixed probe batch.

    model:  the nn.Sequential(TorchLayer, Linear) hybrid model
    X_probe: FIXED probe inputs (same slice every call, for comparability)
    n_total_params: N in the paper reduction (50; classical params
                    contribute zero to the QFIM by definition)

    Implementation note: torch.autograd.functional.jacobian cannot
    differentiate complex outputs, so the state is split into real and
    imaginary parts (view_as_real) and the complex Jacobian is recombined.
    """
    w = model[0].weights.detach().clone().double().requires_grad_(True)
    dim = 2 ** n_qubits
    total = 0.0
    for x in X_probe:
        x64 = x.double()
        # J_ri: real tensor of shape (dim, 2, *w.shape)
        J_ri = torch.autograd.functional.jacobian(
            lambda ww: torch.view_as_real(_state_circuit(x64, ww)), w)
        J_ri = J_ri.reshape(dim, 2, -1)                    # (dim, 2, P)
        Jf = torch.complex(J_ri[:, 0, :], J_ri[:, 1, :])   # (dim, P) complex
        psi = _state_circuit(x64, w.detach())              # (dim,) complex
        overlap = torch.conj(psi) @ Jf                     # <psi|d_i psi>
        gram = torch.sum(torch.conj(Jf) * Jf, dim=0).real  # <d_i psi|d_i psi>
        F_diag = 4.0 * (gram - overlap.abs() ** 2)
        total += F_diag.sum().item()
    return total / len(X_probe) / n_total_params

# --- Input-space QFI: Fubini-Study susceptibility to the encoded input ---
# F_x = sum_i 4( <d_{x_i} psi | d_{x_i} psi> - |<psi | d_{x_i} psi>|^2 )
# This is the quantum-geometric sensitivity of the state to the INPUT,
# routed through the encoding gates (so directly governed by the encoding
# weights omega). RNG-free (analytic backprop).

def input_qfi_scalar(model, X_probe):
    """Mean input-QFI over a fixed probe batch (summed over 4 input dims)."""
    w = model[0].weights.detach().clone().double()
    dim = 2 ** n_qubits
    total = 0.0
    for x in X_probe:
        x64 = x.double().clone().requires_grad_(True)
        J_ri = torch.autograd.functional.jacobian(
            lambda xx: torch.view_as_real(_state_circuit(xx, w)), x64)
        J_ri = J_ri.reshape(dim, 2, -1)                    # (dim, 2, 4)
        Jf = torch.complex(J_ri[:, 0, :], J_ri[:, 1, :])   # (dim, 4) complex
        psi = _state_circuit(x64.detach(), w)              # (dim,) complex
        overlap = torch.conj(psi) @ Jf                     # <psi|d_x psi>
        gram = torch.sum(torch.conj(Jf) * Jf, dim=0).real  # <d_x psi|d_x psi>
        F_x = 4.0 * (gram - overlap.abs() ** 2)            # per input dim
        total += F_x.sum().item()
    return total / len(X_probe)

def encoding_scale_norm(model):
    """Sum of squared encoding scale weights, sum_{l,i} omega_{l,i,0}^2.
    The raw 'frequency selector' magnitude; compare against input-QFI."""
    with torch.no_grad():
        return float((model[0].weights[:, :, 0] ** 2).sum().item())

# --- Main loop ---
print("Experiment 1: diagnostic tracking (STD vs ID, N=50 each)\n")

results = {
    'std':  {'fisher_traj': {s: [] for s in range(SEEDS)}, 'qfim_traj': {s: [] for s in range(SEEDS)}, 'xqfi_traj': {s: [] for s in range(SEEDS)}, 'omega_sq': {s: [] for s in range(SEEDS)}, 'final_acc': []},
    'id':   {'fisher_traj': {s: [] for s in range(SEEDS)}, 'qfim_traj': {s: [] for s in range(SEEDS)}, 'xqfi_traj': {s: [] for s in range(SEEDS)}, 'omega_sq': {s: [] for s in range(SEEDS)}, 'final_acc': []}
}

for seed in range(SEEDS):
    torch.manual_seed(seed)
    np.random.seed(seed)
    
    X, y = generate_data(SAMPLES, FREQ)
    split = int(0.8 * SAMPLES)
    train_loader = DataLoader(TensorDataset(X[:split], y[:split]), batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(TensorDataset(X[split:], y[split:]), batch_size=BATCH_SIZE, shuffle=False)
    X_probe = X[split:split+8]  # fixed probe batch for QFIM (deterministic)
    
    # Initialize both models
    model_std = make_pqc()
    model_id = make_pqc()
    apply_routing_identity_init(model_id)
    
    opt_std = optim.Adam(model_std.parameters(), lr=LR)
    opt_id = optim.Adam(model_id.parameters(), lr=LR)
    criterion = nn.CrossEntropyLoss()
    
    # Track Epoch 0 (Pre-training) Fisher Score
    results['std']['fisher_traj'][seed].append(compute_fisher_score(model_std, val_loader))
    results['id']['fisher_traj'][seed].append(compute_fisher_score(model_id, val_loader))
    results['std']['qfim_traj'][seed].append(qfim_trace_scalar(model_std, X_probe))
    results['id']['qfim_traj'][seed].append(qfim_trace_scalar(model_id, X_probe))
    results['std']['xqfi_traj'][seed].append(input_qfi_scalar(model_std, X_probe))
    results['id']['xqfi_traj'][seed].append(input_qfi_scalar(model_id, X_probe))
    results['std']['omega_sq'][seed].append(encoding_scale_norm(model_std))
    results['id']['omega_sq'][seed].append(encoding_scale_norm(model_id))
    
    # Train
    for epoch in range(1, EPOCHS + 1):
        model_std.train()
        model_id.train()
        
        for bx, by in train_loader:
            # Step Standard PQC
            opt_std.zero_grad()
            loss_std = criterion(model_std(bx), by)
            loss_std.backward()
            opt_std.step()
            
            # Step Identity PQC
            opt_id.zero_grad()
            loss_id = criterion(model_id(bx), by)
            loss_id.backward()
            opt_id.step()
            
        # Track Fisher at milestone epochs
        if epoch in TRACK_EPOCHS:
            results['std']['fisher_traj'][seed].append(compute_fisher_score(model_std, val_loader))
            results['id']['fisher_traj'][seed].append(compute_fisher_score(model_id, val_loader))
            results['std']['qfim_traj'][seed].append(qfim_trace_scalar(model_std, X_probe))
            results['id']['qfim_traj'][seed].append(qfim_trace_scalar(model_id, X_probe))
            results['std']['xqfi_traj'][seed].append(input_qfi_scalar(model_std, X_probe))
            results['id']['xqfi_traj'][seed].append(input_qfi_scalar(model_id, X_probe))
            results['std']['omega_sq'][seed].append(encoding_scale_norm(model_std))
            results['id']['omega_sq'][seed].append(encoding_scale_norm(model_id))

    # Calculate Final Accuracy
    model_std.eval()
    model_id.eval()
    with torch.no_grad():
        acc_std = sum((model_std(bx).argmax(dim=1) == by).sum().item() for bx, by in val_loader) / (SAMPLES - split)
        acc_id = sum((model_id(bx).argmax(dim=1) == by).sum().item() for bx, by in val_loader) / (SAMPLES - split)
        
    results['std']['final_acc'].append(acc_std)
    results['id']['final_acc'].append(acc_id)
    
    # Print Trajectories
    f_std = [f"{f:5.2f}" for f in results['std']['fisher_traj'][seed]]
    f_id = [f"{f:5.2f}" for f in results['id']['fisher_traj'][seed]]
    
    print(f"Seed {seed:02d} | Std Acc: {acc_std:.4f} | Fisher [E0, E10, E30, E60, E120]: {f_std}")
    print(f"        | Id  Acc: {acc_id:.4f} | Fisher [E0, E10, E30, E60, E120]: {f_id}")
    print("-" * 75)

# --- Statistical analysis ---
acc_std_arr = np.array(results['std']['final_acc'])
acc_id_arr = np.array(results['id']['final_acc'])

# Get Final Fisher Scores (index 4 is Epoch 120)
fish_std_final = [results['std']['fisher_traj'][s][4] for s in range(SEEDS)]
fish_id_final = [results['id']['fisher_traj'][s][4] for s in range(SEEDS)]

t_stat_acc, p_val_acc = stats.ttest_ind(acc_id_arr, acc_std_arr, equal_var=False)
t_stat_fish, p_val_fish = stats.ttest_ind(fish_id_final, fish_std_final, equal_var=False)

print("\n--- Final intervention results ---")
print(f"Standard PQC Mean Acc: {np.mean(acc_std_arr):.4f} ± {np.std(acc_std_arr):.4f}")
print(f"Identity PQC Mean Acc: {np.mean(acc_id_arr):.4f} ± {np.std(acc_id_arr):.4f}")
print("-" * 50)
print(f"Standard Final Fisher: {np.mean(fish_std_final):.4f} ± {np.std(fish_std_final):.4f}")
print(f"Identity Final Fisher: {np.mean(fish_id_final):.4f} ± {np.std(fish_id_final):.4f}")
print("-" * 50)
print("WELCH'S T-TEST (Identity vs Standard):")
print(f"Accuracy Shift p-value: {p_val_acc:.4e} (Significant: {p_val_acc < 0.05})")
print(f"Fisher Shift   p-value: {p_val_fish:.4e} (Significant: {p_val_fish < 0.05})")
print("="*50)

# --- Point-biserial correlations against escape outcome ---
from scipy.stats import pointbiserialr

THRESHOLD = 0.65
qfim_std_final = [results['std']['qfim_traj'][s][-1] for s in range(SEEDS)]

binary_escape = (acc_std_arr > THRESHOLD).astype(int)
if 0 < binary_escape.sum() < SEEDS:
    r_fdr, p_fdr = pointbiserialr(binary_escape, np.array(fish_std_final))
    r_qfim, p_qfim = pointbiserialr(binary_escape, np.array(qfim_std_final))
    print("\n--- Point-biserial correlations vs escape (STD) ---")
    print(f"FDR       (fisher_traj): r_pb = {r_fdr:.4f}, p = {p_fdr:.3g}")
    print(f"Param-QFIM (qfim_traj):  r_pb = {r_qfim:.4f}, p = {p_qfim:.3g}")
    xqfi_std_final = [results['std']['xqfi_traj'][s][-1] for s in range(SEEDS)]
    r_xqfi, p_xqfi = pointbiserialr(binary_escape, np.array(xqfi_std_final))
    print(f"Input-QFI (xqfi_traj):   r_pb = {r_xqfi:.4f}, p = {p_xqfi:.3g}")
    from scipy.stats import spearmanr
    om0 = [results['std']['omega_sq'][s][0] for s in range(SEEDS)]
    xq0 = [results['std']['xqfi_traj'][s][0] for s in range(SEEDS)]
    rho, p_rho = spearmanr(om0, xq0)
    print(f"init: Spearman(omega_sq, input-QFI) = {rho:.3f} (p = {p_rho:.3g})")
    r_x0, p_x0 = pointbiserialr(binary_escape, np.array(xq0))
    print(f"init input-QFI vs escape: r_pb = {r_x0:.4f}, p = {p_x0:.3g}")

# Save the dataset for plotting later
with open('experiment_a_results.json', 'w') as f:
    json.dump(results, f, indent=2)

EXPERIMENT A: IDENTITY INTERVENTION & FISHER TRAJECTORY TRACKING

Seed 00 | Std Acc: 0.5300 | Fisher [E0, E10, E30, E60, E120]: [' 0.02', ' 0.01', ' 0.05', ' 0.03', ' 0.02']
        | Id  Acc: 0.5150 | Fisher [E0, E10, E30, E60, E120]: [' 0.01', ' 0.02', ' 0.01', ' 0.00', ' 0.00']
---------------------------------------------------------------------------
Seed 01 | Std Acc: 0.4975 | Fisher [E0, E10, E30, E60, E120]: [' 0.03', ' 0.02', ' 0.02', ' 0.03', ' 0.05']
        | Id  Acc: 0.4587 | Fisher [E0, E10, E30, E60, E120]: [' 0.07', ' 0.03', ' 0.03', ' 0.03', ' 0.03']
---------------------------------------------------------------------------
Seed 02 | Std Acc: 0.8000 | Fisher [E0, E10, E30, E60, E120]: [' 0.01', ' 0.95', ' 1.61', ' 1.82', ' 1.89']
        | Id  Acc: 0.5275 | Fisher [E0, E10, E30, E60, E120]: [' 0.01', ' 0.00', ' 0.02', ' 0.01', ' 0.04']
---------------------------------------------------------------------------
Seed 03 | Std Acc: 0.5100 | Fisher [E0, E10, E30, E60, E12

## Experiment 2 — Routing Ablation (Paper Fig. 2)

Compares three entangler-side conditions:

- **STD**: standard random initialization of both encoding weights (ω) and entangling
  parameters (θ).
- **ID**: entangling layers initialized to approximate the identity, allowing an
  uncorrupted signal to pass through at t=0.
- **COUP**: entangling layers pre-trained via a short independent warmup before joint
  training, attempting to pre-align routing.

If FL were a routing failure, ID or COUP would rescue performance. They do not.

Output: `experiment_b_results_clean.json`.

In [ ]:
import pennylane as qml
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
from scipy import stats
import math
import json

# --- Hyperparameters ---
SAMPLES = 4000
FREQ = 3.0
EPOCHS = 120
BATCH_SIZE = 256
LR = 0.01
SEEDS = 50

n_qubits = 4
n_layers = 2

# --- Model and interventions ---
def generate_data(samples, freq):
    X = (torch.rand(samples, 4) * 2 * math.pi) - math.pi
    y_raw = torch.sin(freq * X[:, 0]) * torch.sin(freq * X[:, 1]) * \
            torch.sin(freq * X[:, 2]) * torch.sin(freq * X[:, 3])
    y = (y_raw > 0).long()
    return X, y

def make_pqc():
    dev = qml.device("default.qubit", wires=n_qubits)
    @qml.qnode(dev, interface="torch")
    def qc(inputs, weights):
        for l in range(n_layers):
            for i in range(n_qubits):
                qml.RX(weights[l, i, 0] * inputs[:, i] + weights[l, i, 1], wires=i)
            qml.StronglyEntanglingLayers(weights[l:l+1, :, 2:5], wires=range(n_qubits))
        return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]
        
    pqc_layer = qml.qnn.TorchLayer(qc, {"weights": (n_layers, n_qubits, 5)})
    return nn.Sequential(pqc_layer, nn.Linear(n_qubits, 2))

def apply_identity_init(model):
    """Initialize entangling angles near zero (approximate identity)."""
    with torch.no_grad():
        for name, param in model.named_parameters():
            if "weights" in name:
                torch.nn.init.normal_(param[:, :, 2:5], mean=0.0, std=0.01)

def apply_coupled_init(model, train_loader):
    """Coupled warmup: pre-align entangling angles to the (frozen) encoding
    weights by maximizing class-centroid separation on a static batch."""
    # Start from the identity initialization
    apply_identity_init(model)
                
    # Build a static batch of ~2000 samples for a stable warmup loss
    bx_list, by_list = [], []
    for bx, by in train_loader:
        bx_list.append(bx)
        by_list.append(by)
        if len(bx_list) * BATCH_SIZE >= 2000:
            break
    static_bx = torch.cat(bx_list)
    static_by = torch.cat(by_list)

    # Backup encoding weights to restore after each step (belt and braces)
    encoding_backup = model[0].weights[:, :, 0:2].detach().clone()
    
    # Optimize full model; encoding-weight gradients are zeroed each step
    opt = optim.Adam(model.parameters(), lr=0.05) 
    
    prev_dist = 0.0
    step = 0
    
    model.train()
    for step in range(50):
        opt.zero_grad()
        out = model[0](static_bx)
        
        out0 = out[static_by == 0]
        out1 = out[static_by == 1]
        
        if len(out0) > 1 and len(out1) > 1:
            mean_0, mean_1 = out0.mean(dim=0), out1.mean(dim=0)
            
            # Objective: maximize class-centroid separation (FDR numerator)
            centroid_dist = ((mean_1 - mean_0) ** 2).sum()
            
            # Convergence check
            if abs(centroid_dist.item() - prev_dist) < 1e-5 and step > 5:
                break
            prev_dist = centroid_dist.item()
            
            loss = -centroid_dist
            loss.backward()
            
            # Zero encoding-weight gradients before the Adam step
            with torch.no_grad():
                if model[0].weights.grad is not None:
                    model[0].weights.grad[:, :, 0:2] = 0.0
            
            opt.step()
            
            # Restore encoding weights exactly (Adam momentum can leak)
            with torch.no_grad():
                model[0].weights[:, :, 0:2] = encoding_backup
                
    return step + 1
    
def compute_fisher(model, val_loader):
    """Classical Fisher discriminant ratio over the validation set."""
    model.eval()
    with torch.no_grad():
        out0_list, out1_list = [], []
        for bx, by in val_loader:
            out = model[0](bx)
            out0_list.append(out[by == 0])
            out1_list.append(out[by == 1])
            
        out0 = torch.cat(out0_list)
        out1 = torch.cat(out1_list)
        
        if len(out0) == 0 or len(out1) == 0:
            return 0.0
            
        mean_0, mean_1 = out0.mean(dim=0), out1.mean(dim=0)
        between = (mean_1 - mean_0) ** 2
        within = (out0.var(dim=0) + out1.var(dim=0)) / 2
        fisher = (between / (within + 0.1)).sum()
        loss = -fisher
    return fisher.item()

# --- Main loop ---
print("Experiment 2: routing ablation (STD / ID / COUP, N=50)\n")

results = {
    'std':  {'f0': [], 'f_final': [], 'acc': []},
    'id':   {'f0': [], 'f_final': [], 'acc': []},
    'coup': {'f_pre': [], 'f0': [], 'f_final': [], 'acc': [], 'warmup_steps': []}
}

for seed in range(SEEDS):
    torch.manual_seed(seed)
    np.random.seed(seed)
    
    X, y = generate_data(SAMPLES, FREQ)
    split = int(0.8 * SAMPLES)
    
    # Independent shuffles per condition
    loaders = {
        'std': DataLoader(TensorDataset(X[:split], y[:split]), batch_size=BATCH_SIZE, shuffle=True),
        'id': DataLoader(TensorDataset(X[:split], y[:split]), batch_size=BATCH_SIZE, shuffle=True),
        'coup': DataLoader(TensorDataset(X[:split], y[:split]), batch_size=BATCH_SIZE, shuffle=True)
    }
    val_loader = DataLoader(TensorDataset(X[split:], y[split:]), batch_size=BATCH_SIZE, shuffle=False)
    
    # Initialize models
    models = {
        'std': make_pqc(),
        'id': make_pqc(),
        'coup': make_pqc()
    }
    
    # Interventions
    apply_identity_init(models['id'])
    
    # Pre-warmup Fisher for COUP
    results['coup']['f_pre'].append(compute_fisher(models['coup'], val_loader))
    
    # COUP warmup
    steps_taken = apply_coupled_init(models['coup'], loaders['coup'])
    results['coup']['warmup_steps'].append(steps_taken)
    
    opts = {k: optim.Adam(m.parameters(), lr=LR) for k, m in models.items()}
    criterion = nn.CrossEntropyLoss()
    
    # Epoch-0 Fisher (post-warmup for COUP)
    for k in models:
        results[k]['f0'].append(compute_fisher(models[k], val_loader))
        
    # Train
    for epoch in range(1, EPOCHS + 1):
        for k, m in models.items():
            m.train()
            
        # Standard
        for bx, by in loaders['std']:
            opts['std'].zero_grad()
            loss = criterion(models['std'](bx), by)
            loss.backward()
            opts['std'].step()
            
        # Identity
        for bx, by in loaders['id']:
            opts['id'].zero_grad()
            loss = criterion(models['id'](bx), by)
            loss.backward()
            opts['id'].step()
            
        # Coupled
        for bx, by in loaders['coup']:
            opts['coup'].zero_grad()
            loss = criterion(models['coup'](bx), by)
            loss.backward()
            opts['coup'].step()
                
    # Final metrics
    for k, m in models.items():
        m.eval()
        with torch.no_grad():
            acc = sum((m(bx).argmax(dim=1) == by).sum().item() for bx, by in val_loader) / (SAMPLES - split)
        results[k]['acc'].append(acc)
        results[k]['f_final'].append(compute_fisher(m, val_loader))
        
    print(f"Seed {seed:02d} | Coup Warmup: {steps_taken} steps")
    print(f"        | E0 Fisher -> Std: {results['std']['f0'][-1]:.3f} | Id: {results['id']['f0'][-1]:.3f} | Coup(Pre): {results['coup']['f_pre'][-1]:.3f} -> Coup(Post): {results['coup']['f0'][-1]:.3f}")
    print(f"        | Final Acc -> Std: {results['std']['acc'][-1]:.4f} | Id: {results['id']['acc'][-1]:.4f} | Coup: {results['coup']['acc'][-1]:.4f}")
    print("-" * 100)

# --- Statistical analysis ---
mean_acc = {k: np.mean(results[k]['acc']) for k in results}
std_acc = {k: np.std(results[k]['acc']) for k in results}
mean_f0 = {k: np.mean(results[k]['f0']) for k in results}

_, p_coup_vs_std = stats.ttest_ind(results['coup']['acc'], results['std']['acc'], equal_var=False)
_, p_coup_vs_id = stats.ttest_ind(results['coup']['acc'], results['id']['acc'], equal_var=False)

print("\n--- Final intervention results ---")
print(f"Standard PQC Mean Acc: {mean_acc['std']:.4f} ± {std_acc['std']:.4f} (E0 Fisher: {mean_f0['std']:.3f})")
print(f"Identity PQC Mean Acc: {mean_acc['id']:.4f}   ± {std_acc['id']:.4f} (E0 Fisher: {mean_f0['id']:.3f})")
print(f"Coupled  PQC Mean Acc: {mean_acc['coup']:.4f} ± {std_acc['coup']:.4f} (E0 Fisher: {mean_f0['coup']:.3f})")
print("Welch t-test (accuracy):")
print(f"Coupled vs Standard p-value: {p_coup_vs_std:.4e}")
print(f"Coupled vs Identity p-value: {p_coup_vs_id:.4e}")
print()

with open('experiment_b_results_clean.json', 'w') as f:
    json.dump(results, f, indent=2)

## Experiment 3 — Curriculum ablation (Fig. 3, Fig. 4)

Four training curricula, each 120 epochs split into three 40-epoch phases:

- **STD** — all three phases at $f=3.0$ (control).
- **FWD** — $f=1.0 \to 2.0 \to 3.0$ (spectral homotopy).
- **REV** — $f=3.0 \to 2.0 \to 1.0$.
- **RND** — shuffled phase order per seed.

Train/val split 3200/800. Per-phase target-accuracy checkpoints under
`phase_accs`.

At each phase boundary the parameter-space QFIM scalar
$\frac{1}{N}\mathrm{Tr}(F)$ (`phase_qfi`) and the input-space QFI $F_x$
(`phase_xqfi`) are evaluated analytically on a fixed probe batch (first 8
validation inputs). Both consume no RNG; the reproduction gate in the next
cell verifies this by checking that final accuracies match a previous run
without logging.

Output: `experiment_f_bulletproof_qfi.json`.

In [ ]:
import pennylane as qml
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import math
import json

# --- Settings and model ---
TRAIN_SAMPLES = 3200
VAL_SAMPLES = 800
TARGET_FREQ = 3.0
PHASE_EPOCHS = 40 
BATCH_SIZE = 256
LR = 0.01
SEEDS = 50 

n_qubits = 4
n_layers = 2

def generate_data(samples, freq):
    X = (torch.rand(samples, 4) * 2 * math.pi) - math.pi
    y_raw = torch.sin(freq * X[:, 0]) * torch.sin(freq * X[:, 1]) * \
            torch.sin(freq * X[:, 2]) * torch.sin(freq * X[:, 3])
    y = (y_raw > 0).long()
    return X, y

def make_pqc():
    dev = qml.device("default.qubit", wires=n_qubits)
    @qml.qnode(dev, interface="torch")
    def qc(inputs, weights):
        for l in range(n_layers):
            for i in range(n_qubits):
                qml.RX(weights[l, i, 0] * inputs[:, i] + weights[l, i, 1], wires=i)
            qml.StronglyEntanglingLayers(weights[l:l+1, :, 2:5], wires=range(n_qubits))
        return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]
    pqc_layer = qml.qnn.TorchLayer(qc, {"weights": (n_layers, n_qubits, 5)})
    return nn.Sequential(pqc_layer, nn.Linear(n_qubits, 2))

# --- Parameter-space QFIM: 1/N Tr(F), paper Eq. (2) ---
# F_ii = 4(<d_i psi|d_i psi> - |<psi|d_i psi>|^2), analytic backprop, RNG-free.

_qfim_dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(_qfim_dev, interface="torch", diff_method="backprop")
def _state_circuit(x, weights):
    for l in range(n_layers):
        for i in range(n_qubits):
            qml.RX(weights[l, i, 0] * x[i] + weights[l, i, 1], wires=i)
        qml.StronglyEntanglingLayers(weights[l:l+1, :, 2:5], wires=range(n_qubits))
    return qml.state()

def qfim_trace_scalar(model, X_probe, n_total_params=50):
    """Normalized QFIM trace (1/N Tr(F)) averaged over a fixed probe batch.

    model:  the nn.Sequential(TorchLayer, Linear) hybrid model
    X_probe: FIXED probe inputs (same slice every call, for comparability)
    n_total_params: N in the paper reduction (50; classical params
                    contribute zero to the QFIM by definition)

    Implementation note: torch.autograd.functional.jacobian cannot
    differentiate complex outputs, so the state is split into real and
    imaginary parts (view_as_real) and the complex Jacobian is recombined.
    """
    w = model[0].weights.detach().clone().double().requires_grad_(True)
    dim = 2 ** n_qubits
    total = 0.0
    for x in X_probe:
        x64 = x.double()
        # J_ri: real tensor of shape (dim, 2, *w.shape)
        J_ri = torch.autograd.functional.jacobian(
            lambda ww: torch.view_as_real(_state_circuit(x64, ww)), w)
        J_ri = J_ri.reshape(dim, 2, -1)                    # (dim, 2, P)
        Jf = torch.complex(J_ri[:, 0, :], J_ri[:, 1, :])   # (dim, P) complex
        psi = _state_circuit(x64, w.detach())              # (dim,) complex
        overlap = torch.conj(psi) @ Jf                     # <psi|d_i psi>
        gram = torch.sum(torch.conj(Jf) * Jf, dim=0).real  # <d_i psi|d_i psi>
        F_diag = 4.0 * (gram - overlap.abs() ** 2)
        total += F_diag.sum().item()
    return total / len(X_probe) / n_total_params

# --- Input-space QFI: F_x = sum_i 4(<d_{x_i} psi|d_{x_i} psi> - |<psi|d_{x_i} psi>|^2) ---
# Governed by the encoding weights omega. Analytic backprop, RNG-free.

def input_qfi_scalar(model, X_probe):
    """Mean input-QFI over a fixed probe batch (summed over 4 input dims)."""
    w = model[0].weights.detach().clone().double()
    dim = 2 ** n_qubits
    total = 0.0
    for x in X_probe:
        x64 = x.double().clone().requires_grad_(True)
        J_ri = torch.autograd.functional.jacobian(
            lambda xx: torch.view_as_real(_state_circuit(xx, w)), x64)
        J_ri = J_ri.reshape(dim, 2, -1)                    # (dim, 2, 4)
        Jf = torch.complex(J_ri[:, 0, :], J_ri[:, 1, :])   # (dim, 4) complex
        psi = _state_circuit(x64.detach(), w)              # (dim,) complex
        overlap = torch.conj(psi) @ Jf                     # <psi|d_x psi>
        gram = torch.sum(torch.conj(Jf) * Jf, dim=0).real  # <d_x psi|d_x psi>
        F_x = 4.0 * (gram - overlap.abs() ** 2)            # per input dim
        total += F_x.sum().item()
    return total / len(X_probe)

# --- Main loop ---
print("Experiment 3: curriculum ablation (STD / FWD / REV / RND, N=50)\n")

# Base conditions setup
base_conditions = ['std', 'fwd', 'rev', 'rnd']
results = {cond: {'final_acc': [], 'phase_accs': {str(s): [] for s in range(SEEDS)}, 'phase_qfi': {str(s): [] for s in range(SEEDS)}, 'phase_xqfi': {str(s): [] for s in range(SEEDS)}} for cond in base_conditions}

for seed in range(SEEDS):
    torch.manual_seed(seed)
    np.random.seed(seed)
    
    # Fresh permutation per seed for RND
    rnd_order = [1.0, 2.0, 3.0]
    np.random.shuffle(rnd_order)
    
    active_conditions = {
        'std': [3.0, 3.0, 3.0], 
        'fwd': [1.0, 2.0, 3.0], 
        'rev': [3.0, 2.0, 1.0], 
        'rnd': rnd_order  
    }
    
    # 1 Static Validation Set for fair evaluation
    X_val, y_val = generate_data(VAL_SAMPLES, TARGET_FREQ)
    val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=BATCH_SIZE, shuffle=False)
    X_probe = X_val[:8]  # fixed probe batch for QFIM (deterministic, no RNG)
    
    # Shared initial state across the four conditions
    master_model = make_pqc()
    init_state = master_model.state_dict()
    
    models = {cond: make_pqc() for cond in active_conditions}
    opts = {}
    for cond in active_conditions:
        models[cond].load_state_dict(init_state)
        opts[cond] = optim.Adam(models[cond].parameters(), lr=LR)
        
    criterion = nn.CrossEntropyLoss()
    
    # Train Phase by Phase
    for phase_idx in range(3):
        
        # Fresh 3200-sample dataset per condition per phase
        phase_loaders = {}
        for cond, freqs in active_conditions.items():
            freq = freqs[phase_idx]
            X_train, y_train = generate_data(TRAIN_SAMPLES, freq)
            phase_loaders[cond] = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
            
        for epoch in range(PHASE_EPOCHS):
            for cond in active_conditions:
                models[cond].train()
                for bx, by in phase_loaders[cond]:
                    opts[cond].zero_grad()
                    loss = criterion(models[cond](bx), by)
                    loss.backward()
                    opts[cond].step()
                    
        # Evaluate TARGET accuracy at the end of this phase
        for cond in active_conditions:
            models[cond].eval()
            with torch.no_grad():
                acc = sum((models[cond](bx).argmax(dim=1) == by).sum().item() for bx, by in val_loader) / VAL_SAMPLES
            results[cond]['phase_accs'][str(seed)].append(acc)  # str key for JSON
            # Parameter-space QFIM at the phase boundary (RNG-free)
            q = qfim_trace_scalar(models[cond], X_probe)
            results[cond]['phase_qfi'][str(seed)].append(q)
            # Input-space F_x at the phase boundary (RNG-free)
            xq = input_qfi_scalar(models[cond], X_probe)
            results[cond]['phase_xqfi'][str(seed)].append(xq)
            
            if phase_idx == 2:
                results[cond]['final_acc'].append(acc)

    if (seed + 1) % 5 == 0 or seed == 0:
        print(f"Completed Seed {seed + 1}/{SEEDS}...")

# Save the dataset
with open('experiment_f_bulletproof_qfi.json', 'w') as f:
    json.dump(results, f, indent=2)

# --- Escape-rate sensitivity to the threshold ---
print("\n--- Escape rates by threshold ---")

thresholds = [0.60, 0.65, 0.70]

for cond in base_conditions:
    accs = np.array(results[cond]['final_acc'])
    print(f"\n--- {cond.upper()} Condition ---")
    print(f"Mean (All): {np.mean(accs):.4f}")
    
    for thresh in thresholds:
        success_rate = np.mean(accs >= thresh) * 100
        print(f"  Escape Rate (> {thresh:.2f}) : {success_rate:.1f}%")

## Reproduction gate

Verifies that the analytic QFI evaluation consumed no RNG: the re-run's
`final_acc` arrays must match the previously published
`experiment_f_bulletproof.json` exactly. On success, `phase_qfi` and
`phase_xqfi` are merged into the canonical file. If any condition diverges,
the file is not merged and the discrepancy must be traced.

In [ ]:
import json
import numpy as np

old = json.load(open('experiment_f_bulletproof.json'))
new = json.load(open('experiment_f_bulletproof_qfi.json'))

all_ok = True
for cond in ['std', 'fwd', 'rev', 'rnd']:
    a_old = np.array(old[cond]['final_acc'])
    a_new = np.array(new[cond]['final_acc'])
    ok = a_old.shape == a_new.shape and np.allclose(a_old, a_new, atol=1e-9)
    all_ok &= ok
    print(f"{cond.upper():4s} final_acc identical: {ok}")
    if not ok:
        diffs = np.where(~np.isclose(a_old, a_new, atol=1e-9))[0]
        print(f"     diverging seeds: {diffs.tolist()[:10]}")

assert all_ok, "Reproduction FAILED - do not merge; debug RNG perturbation."
print("\nAll conditions reproduce exactly.")

# --- Merge: add phase_qfi into the canonical file ---
for cond in ['std', 'fwd', 'rev', 'rnd']:
    old[cond]['phase_qfi'] = new[cond]['phase_qfi']
    if 'phase_xqfi' in new[cond]:
        old[cond]['phase_xqfi'] = new[cond]['phase_xqfi']
with open('experiment_f_bulletproof.json', 'w') as f:
    json.dump(old, f, indent=2)
print("\nMerged phase_qfi and phase_xqfi into experiment_f_bulletproof.json.")

## Experiment 4 — Fisher exact tests (Sec. V.B)

Fisher exact tests on the 50-seed contingency tables from Experiment 3,
comparing each curriculum against the STD baseline.

In [ ]:
import json
import numpy as np
from scipy.stats import fisher_exact

with open('experiment_f_bulletproof.json') as f:
    results = json.load(f)

THRESHOLD = 0.65
SEEDS = 50

def counts(cond):
    accs = np.array(results[cond]['final_acc'])
    esc = int((accs > THRESHOLD).sum())
    return esc, SEEDS - esc

std_e, std_f = counts('std')
fwd_e, fwd_f = counts('fwd')
rev_e, rev_f = counts('rev')
rnd_e, rnd_f = counts('rnd')

print(f"Escape counts (threshold {THRESHOLD}):")
print(f"  STD {std_e}/{SEEDS}   FWD {fwd_e}/{SEEDS}   REV {rev_e}/{SEEDS}   RND {rnd_e}/{SEEDS}")

_, p_fwd = fisher_exact([[fwd_e, fwd_f], [std_e, std_f]], alternative='greater')
_, p_rev = fisher_exact([[rev_e, rev_f], [std_e, std_f]], alternative='less')
_, p_rnd = fisher_exact([[rnd_e, rnd_f], [std_e, std_f]], alternative='greater')

print("\nFisher exact vs. STD (one-sided):")
print(f"  FWD: p = {p_fwd:.4f}")
print(f"  REV: p = {p_rev:.4f}")
print(f"  RND: p = {p_rnd:.4f}")
print("\nEach contrast is evaluated at alpha = 0.05 (one-sided, uncorrected), as in the paper.")

## RND order reconstruction (Sec. V.B stratified analysis)

Each seed's RND phase order is drawn by `np.random.shuffle` immediately after
`np.random.seed(seed)`, so the per-seed orderings are exactly reconstructible.
Stratifying RND escapes by the drawn ordering provides an internal test of the
ordering hypothesis.

In [4]:
import json
import numpy as np
from scipy.stats import fisher_exact

with open('experiment_f_bulletproof.json') as f:
    results = json.load(f)

SEEDS = 50
THRESHOLD = 0.65
rnd_acc = np.array(results['rnd']['final_acc'])
escaped = rnd_acc > THRESHOLD

# Reconstruct each seed's phase order (matches the Experiment 3 loop exactly)
orders = {}
for seed in range(SEEDS):
    np.random.seed(seed)
    order = [1.0, 2.0, 3.0]
    np.random.shuffle(order)
    orders[seed] = tuple(order)

fwd = (1.0, 2.0, 3.0)
acc_fwd = [s for s in range(SEEDS) if orders[s] == fwd]
print(f"Seeds that drew the exact forward order: {len(acc_fwd)} -> {acc_fwd}")

print("\nRND escapes and their drawn orders:")
for s in np.where(escaped)[0]:
    print(f"  seed {s:2d}: {orders[s]}  final acc {rnd_acc[s]:.4f}")

n_af_esc = sum(1 for s in np.where(escaped)[0] if orders[s] == fwd)
n_af = len(acc_fwd)
n_other_esc = int(escaped.sum()) - n_af_esc
n_other = SEEDS - n_af
print(f"\nAccidental-forward escape rate: {n_af_esc}/{n_af}")
print(f"Other-ordering escape rate:     {n_other_esc}/{n_other}")
_, p1 = fisher_exact([[n_af_esc, n_af - n_af_esc],
                      [n_other_esc, n_other - n_other_esc]], alternative='greater')
print(f"Fisher exact (one-sided): p = {p1:.4f}")

end3 = [s for s in range(SEEDS) if orders[s][-1] == 3.0]
e_in = sum(1 for s in np.where(escaped)[0] if s in end3)
print(f"\nOrders ending at the target frequency: {len(end3)} seeds, escapes {e_in}/{len(end3)}")
print(f"Orders not ending at the target:        {SEEDS-len(end3)} seeds, escapes {int(escaped.sum())-e_in}/{SEEDS-len(end3)}")
_, p2 = fisher_exact([[e_in, len(end3)-e_in],
                      [int(escaped.sum())-e_in, SEEDS-len(end3)-(int(escaped.sum())-e_in)]],
                     alternative='greater')
print(f"Fisher exact (one-sided): p = {p2:.4f}")

Seeds that drew the exact forward order: 12 -> [5, 6, 12, 18, 20, 24, 31, 40, 42, 45, 47, 49]

RND escapes and their drawn orders:
  seed  3: (2.0, 1.0, 3.0)  final acc 0.9087
  seed  5: (1.0, 2.0, 3.0)  final acc 0.7025
  seed 24: (1.0, 2.0, 3.0)  final acc 0.8975
  seed 49: (1.0, 2.0, 3.0)  final acc 0.7662

Accidental-forward escape rate: 3/12
Other-ordering escape rate:     1/38
Fisher exact (one-sided): p = 0.0384

Orders ending at the target frequency: 17 seeds, escapes 4/17
Orders not ending at the target:        33 seeds, escapes 0/33
Fisher exact (one-sided): p = 0.0103


## Experiment 5 — FWD trajectory analysis (Fig. 4, Sec. V.C)

Per-phase target-accuracy trajectories for the successful FWD seeds and the
Phase-1 fitters under REV — the mechanistic content of Fig. 4 and the
spectral-forgetting analysis.

In [ ]:
import json
import numpy as np

with open('experiment_f_bulletproof.json') as f:
    results = json.load(f)

SEEDS = 50
THRESHOLD = 0.65

# --- FWD success trajectories ---
print("--- Successful FWD trajectories ---")
fwd_success = []
for s in range(SEEDS):
    traj = results['fwd']['phase_accs'][str(s)]
    if traj[-1] > THRESHOLD:
        fwd_success.append(traj)

if fwd_success:
    mean_traj = np.mean(fwd_success, axis=0)
    print(f"Mean of {len(fwd_success)} successful FWD seeds:")
    print(f"  Phase 1 (f=1.0): {mean_traj[0]:.4f}")
    print(f"  Phase 2 (f=2.0): {mean_traj[1]:.4f}")
    print(f"  Phase 3 (f=3.0): {mean_traj[2]:.4f}")
    print("\nPer-seed trajectories:")
    for i, t in enumerate(fwd_success):
        print(f"  seed {i+1:02d}: {t[0]:.4f} -> {t[1]:.4f} -> {t[2]:.4f}")
else:
    print("No FWD successes.")

# --- REV Phase-1 fitters: spectral forgetting ---
print("\n--- REV Phase-1 fitters (spectral forgetting) ---")
rev_p1 = [results['rev']['phase_accs'][str(s)][0] for s in range(SEEDS)]
p1_fit = sum(1 for a in rev_p1 if a > THRESHOLD)
print(f"REV Phase-1 fit rate: {p1_fit}/{SEEDS}")

if p1_fit > 0:
    print("Trajectories of seeds that fit in Phase 1 and were then destroyed:")
    for s in range(SEEDS):
        traj = results['rev']['phase_accs'][str(s)]
        if traj[0] > THRESHOLD:
            print(f"  seed {s:02d}: {traj[0]:.4f} -> {traj[1]:.4f} -> {traj[2]:.4f}")